In [ ]:
"""
BLUESTOCK MF - DAY 2 PIPELINE

Day-2 requirements:
1. Clean nav_history.csv
2. Clean investor_transactions.csv
3. Clean scheme_performance.csv
4. Design SQLite star schema
5. Load cleaned datasets into SQLite
6. Verify row counts
7. Write 10 analytical SQL queries
8. Write data dictionary
9. Produce required processed CSV files

Run from the project root:

    python day2_pipeline.py
"""

from pathlib import Path
import sqlite3
import sys
import traceback
import re
import shutil

import pandas as pd
from sqlalchemy import create_engine


# ================================================================
# 1. PROJECT PATHS
# ================================================================

def find_project_root():

    candidates = []

    if "__file__" in globals():
        candidates.append(
            Path(__file__).resolve().parent
        )

    candidates.append(Path.cwd())

    for candidate in candidates:

        candidate = candidate.resolve()

        if (candidate / "data" / "raw").exists():
            return candidate

    return candidates[0].resolve()


BASE = find_project_root()

RAW = BASE / "data" / "raw"
PROCESSED = BASE / "data" / "processed"

DB_PATH = BASE / "bluestock_mf.db"
SCHEMA_PATH = BASE / "schema.sql"
QUERIES_PATH = BASE / "queries.sql"
DICTIONARY_PATH = BASE / "data_dictionary.md"


# ================================================================
# 2. SOURCE FILES
# ================================================================

SOURCE_FILES = [
    "01_fund_master.csv",
    "02_nav_history.csv",
    "03_aum_by_fund_house.csv",
    "04_monthly_sip_inflows.csv",
    "05_category_inflows.csv",
    "06_industry_folio_count.csv",
    "07_scheme_performance.csv",
    "08_investor_transactions.csv",
    "09_portfolio_holdings.csv",
    "10_benchmark_indices.csv",
]


# ================================================================
# 3. GENERAL HELPERS
# ================================================================

def normalize_name(value):

    return (
        str(value)
        .strip()
        .lower()
        .replace(" ", "")
        .replace("_", "")
        .replace("-", "")
        .replace(".", "")
        .replace("/", "")
    )


def find_column(
    df,
    aliases,
    required=True,
    label=""
):

    lookup = {
        normalize_name(column): column
        for column in df.columns
    }

    for alias in aliases:

        key = normalize_name(alias)

        if key in lookup:
            return lookup[key]

    if required:

        raise KeyError(
            "\nRequired column not found.\n"
            f"Tried: {aliases}\n"
            f"Available columns: {list(df.columns)}"
        )

    return None


def clean_code_series(series):

    return (
        series
        .astype(str)
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
    )


def parse_dates(series):

    values = (
        series
        .astype(str)
        .str.strip()
    )

    parsed = pd.to_datetime(
        values,
        errors="coerce",
        dayfirst=True
    )

    # Try month-first for values where day-first failed.
    missing = parsed.isna()

    if missing.any():

        alternative = pd.to_datetime(
            values[missing],
            errors="coerce",
            dayfirst=False
        )

        parsed.loc[missing] = alternative

    return parsed


def print_report(name, df):

    print()
    print(f"[{name}]")
    print(f"Rows    : {len(df):,}")
    print(f"Columns : {len(df.columns)}")
    print(f"Columns : {list(df.columns)}")


def read_csv(filename):

    path = RAW / filename

    if not path.exists():

        raise FileNotFoundError(
            f"Missing file: {path}"
        )

    print(
        f"[READ] {filename}"
    )

    return pd.read_csv(path)


def write_csv(df, filename):

    PROCESSED.mkdir(
        parents=True,
        exist_ok=True
    )

    path = PROCESSED / filename

    df.to_csv(
        path,
        index=False
    )

    print(
        f"[WRITE] {path}"
    )


# ================================================================
# 4. PROJECT CHECK
# ================================================================

def verify_project():

    print("=" * 70)
    print("BLUESTOCK MF - DAY 2 PIPELINE")
    print("=" * 70)

    print()
    print("=" * 70)
    print("PROJECT CHECK")
    print("=" * 70)

    print(
        f"Project root : {BASE}"
    )

    print(
        f"Raw data     : {RAW}"
    )

    print(
        f"Processed    : {PROCESSED}"
    )

    print()

    if not RAW.exists():

        raise FileNotFoundError(
            f"Raw folder does not exist: {RAW}"
        )

    PROCESSED.mkdir(
        parents=True,
        exist_ok=True
    )

    missing = []

    for filename in SOURCE_FILES:

        path = RAW / filename

        if path.exists():

            print(
                f"[OK] {filename}"
            )

        else:

            print(
                f"[MISSING] {filename}"
            )

            missing.append(filename)

    print()

    if missing:

        raise FileNotFoundError(
            "Missing required datasets:\n"
            + "\n".join(
                f" - {x}"
                for x in missing
            )
        )

    print(
        "[OK] All 10 datasets found."
    )


# ================================================================
# 5. NAV HISTORY CLEANING
# ================================================================

def clean_nav_history():

    print()
    print("=" * 70)
    print("1. CLEANING 02_nav_history.csv")
    print("=" * 70)

    df = read_csv(
        "02_nav_history.csv"
    )

    print_report(
        "NAV RAW",
        df
    )

    col_amfi = find_column(
        df,
        [
            "amfi_code",
            "scheme_code",
            "amficode",
            "code"
        ],
        label="amfi_code"
    )

    col_date = find_column(
        df,
        [
            "date",
            "nav_date",
            "navdate",
            "as_of_date"
        ],
        label="date"
    )

    col_nav = find_column(
        df,
        [
            "nav",
            "net_asset_value",
            "nav_value"
        ],
        label="nav"
    )

    df = df.rename(
        columns={
            col_amfi: "amfi_code",
            col_date: "date",
            col_nav: "nav"
        }
    )

    df = df[
        [
            "amfi_code",
            "date",
            "nav"
        ]
    ].copy()

    df["amfi_code"] = clean_code_series(
        df["amfi_code"]
    )

    df["date"] = parse_dates(
        df["date"]
    )

    df["nav"] = pd.to_numeric(
        df["nav"],
        errors="coerce"
    )

    # ------------------------------------------------------------
    # Reject invalid dates
    # ------------------------------------------------------------

    bad_date = df[
        df["date"].isna()
    ].copy()

    bad_date["reject_reason"] = (
        "invalid date"
    )

    df = df.dropna(
        subset=["date"]
    )

    # ------------------------------------------------------------
    # Reject invalid NAV
    # ------------------------------------------------------------

    bad_nav = df[
        df["nav"].isna() |
        (df["nav"] <= 0)
    ].copy()

    bad_nav["reject_reason"] = (
        "NAV must be numeric and greater than 0"
    )

    df = df[
        df["nav"].notna() &
        (df["nav"] > 0)
    ]

    # ------------------------------------------------------------
    # Sort before duplicate handling / forward fill
    # ------------------------------------------------------------

    df = df.sort_values(
        [
            "amfi_code",
            "date"
        ]
    ).reset_index(
        drop=True
    )

    # ------------------------------------------------------------
    # Remove duplicate fund/date records
    # ------------------------------------------------------------

    duplicate_mask = df.duplicated(
        subset=[
            "amfi_code",
            "date"
        ],
        keep="last"
    )

    duplicates = df[
        duplicate_mask
    ].copy()

    duplicates["reject_reason"] = (
        "duplicate amfi_code/date"
    )

    df = df[
        ~duplicate_mask
    ].copy()

    # ------------------------------------------------------------
    # Forward-fill missing NAV within each fund.
    #
    # This applies to missing NAV values that remain after
    # numeric validation. Existing valid NAV observations are
    # carried forward by fund.
    # ------------------------------------------------------------

    df["nav"] = (
        df
        .groupby("amfi_code")["nav"]
        .ffill()
    )

    # ------------------------------------------------------------
    # Remove rows that still have no NAV
    # ------------------------------------------------------------

    still_missing = df[
        df["nav"].isna()
    ].copy()

    still_missing["reject_reason"] = (
        "NAV missing and no previous NAV available"
    )

    df = df.dropna(
        subset=["nav"]
    )

    # ------------------------------------------------------------
    # Save
    # ------------------------------------------------------------

    rejects = pd.concat(
        [
            bad_date,
            bad_nav,
            duplicates,
            still_missing
        ],
        ignore_index=True
    )

    if not rejects.empty:

        write_csv(
            rejects,
            "02_nav_history_rejects.csv"
        )

    write_csv(
        df,
        "02_nav_history_clean.csv"
    )

    print_report(
        "NAV CLEAN",
        df
    )

    print(
        f"[NAV] rejected rows: {len(rejects):,}"
    )

    return df


# ================================================================
# 6. INVESTOR TRANSACTIONS CLEANING
# ================================================================

TRANSACTION_TYPE_MAP = {

    "sip": "SIP",
    "systematicinvestmentplan": "SIP",
    "systematicinvestment": "SIP",

    "lumpsum": "Lumpsum",
    "lumpsuminvestment": "Lumpsum",
    "onetime": "Lumpsum",
    "onetimeinvestment": "Lumpsum",
    "purchase": "Lumpsum",
    "buy": "Lumpsum",

    "redemption": "Redemption",
    "redeem": "Redemption",
    "withdrawal": "Redemption",
    "sell": "Redemption",
}


VALID_KYC = {
    "VERIFIED",
    "PENDING",
    "REJECTED",
    "NOT_INITIATED"
}


def clean_investor_transactions():

    print()
    print("=" * 70)
    print("2. CLEANING 08_investor_transactions.csv")
    print("=" * 70)

    df = read_csv(
        "08_investor_transactions.csv"
    )

    print_report(
        "TRANSACTIONS RAW",
        df
    )

    col_transaction_id = find_column(
        df,
        [
            "transaction_id",
            "txn_id",
            "transactionid",
            "id"
        ],
        required=False,
        label="transaction_id"
    )

    col_investor = find_column(
        df,
        [
            "investor_id",
            "investorid",
            "folio_no",
            "folio_number",
            "client_id"
        ],
        label="investor_id"
    )

    col_amfi = find_column(
        df,
        [
            "amfi_code",
            "scheme_code",
            "amficode",
            "code"
        ],
        label="amfi_code"
    )

    col_date = find_column(
        df,
        [
            "date",
            "txn_date",
            "transaction_date",
            "transactiondate"
        ],
        label="date"
    )

    col_type = find_column(
        df,
        [
            "transaction_type",
            "txn_type",
            "transactiontype",
            "type"
        ],
        label="transaction_type"
    )

    col_amount = find_column(
        df,
        [
            "amount",
            "txn_amount",
            "transaction_amount",
            "value",
            "amount_inr"
        ],
        label="amount"
    )

    col_kyc = find_column(
        df,
        [
            "kyc_status",
            "kyc",
            "kycstatus"
        ],
        label="kyc_status"
    )

    col_state = find_column(
        df,
        [
            "state",
            "investor_state"
        ],
        required=False,
        label="state"
    )

    rename = {
        col_investor: "investor_id",
        col_amfi: "amfi_code",
        col_date: "date",
        col_type: "transaction_type",
        col_amount: "amount",
        col_kyc: "kyc_status"
    }

    if col_transaction_id:
        rename[col_transaction_id] = "transaction_id"

    if col_state:
        rename[col_state] = "state"

    df = df.rename(
        columns=rename
    )

    # ------------------------------------------------------------
    # Transaction ID
    # ------------------------------------------------------------

    if "transaction_id" not in df.columns:

        df = df.reset_index(
            drop=True
        )

        df["transaction_id"] = (
            "TXN"
            + (df.index + 1)
            .astype(str)
            .str.zfill(7)
        )

    # ------------------------------------------------------------
    # Keep required columns
    # ------------------------------------------------------------

    keep = [
        "transaction_id",
        "investor_id",
        "amfi_code",
        "date",
        "transaction_type",
        "amount",
        "kyc_status"
    ]

    if "state" in df.columns:
        keep.append("state")

    df = df[
        keep
    ].copy()

    # ------------------------------------------------------------
    # Basic string cleaning
    # ------------------------------------------------------------

    df["transaction_id"] = (
        df["transaction_id"]
        .astype(str)
        .str.strip()
    )

    df["investor_id"] = (
        df["investor_id"]
        .astype(str)
        .str.strip()
    )

    df["amfi_code"] = clean_code_series(
        df["amfi_code"]
    )

    # ------------------------------------------------------------
    # Transaction type
    # ------------------------------------------------------------

    original_transaction_type = (
        df["transaction_type"]
        .astype(str)
        .str.strip()
        .map(normalize_name)
    )

    df["transaction_type"] = (
        original_transaction_type
        .map(TRANSACTION_TYPE_MAP)
    )

    bad_type = df[
        df["transaction_type"].isna()
    ].copy()

    bad_type["reject_reason"] = (
        "unrecognised transaction_type"
    )

    df = df[
        df["transaction_type"].notna()
    ].copy()

    # ------------------------------------------------------------
    # Amount
    # ------------------------------------------------------------

    df["amount"] = pd.to_numeric(
        df["amount"],
        errors="coerce"
    )

    bad_amount = df[
        df["amount"].isna() |
        (df["amount"] <= 0)
    ].copy()

    bad_amount["reject_reason"] = (
        "amount must be numeric and greater than 0"
    )

    df = df[
        df["amount"].notna() &
        (df["amount"] > 0)
    ].copy()

    # ------------------------------------------------------------
    # Dates
    # ------------------------------------------------------------

    df["date"] = parse_dates(
        df["date"]
    )

    bad_date = df[
        df["date"].isna()
    ].copy()

    bad_date["reject_reason"] = (
        "invalid date"
    )

    df = df.dropna(
        subset=["date"]
    )

    # ------------------------------------------------------------
    # KYC status
    # ------------------------------------------------------------

    df["kyc_status"] = (
        df["kyc_status"]
        .astype(str)
        .str.strip()
        .str.upper()
        .str.replace(
            " ",
            "_"
        )
    )

    df["kyc_status"] = (
        df["kyc_status"]
        .replace(
            {
                "NOTINITIATED": "NOT_INITIATED"
            }
        )
    )

    bad_kyc = df[
        ~df["kyc_status"].isin(
            VALID_KYC
        )
    ].copy()

    bad_kyc["reject_reason"] = (
        "invalid kyc_status"
    )

    df = df[
        df["kyc_status"].isin(
            VALID_KYC
        )
    ].copy()

    # ------------------------------------------------------------
    # Duplicate transaction IDs
    # ------------------------------------------------------------

    duplicate_mask = df.duplicated(
        subset=["transaction_id"],
        keep="last"
    )

    duplicates = df[
        duplicate_mask
    ].copy()

    duplicates["reject_reason"] = (
        "duplicate transaction_id"
    )

    df = df[
        ~duplicate_mask
    ].copy()

    # ------------------------------------------------------------
    # Sort
    # ------------------------------------------------------------

    df = df.sort_values(
        [
            "date",
            "transaction_id"
        ]
    ).reset_index(
        drop=True
    )

    # ------------------------------------------------------------
    # Save
    # ------------------------------------------------------------

    rejects = pd.concat(
        [
            bad_type,
            bad_amount,
            bad_date,
            bad_kyc,
            duplicates
        ],
        ignore_index=True
    )

    if not rejects.empty:

        write_csv(
            rejects,
            "08_investor_transactions_rejects.csv"
        )

    write_csv(
        df,
        "08_investor_transactions_clean.csv"
    )

    print_report(
        "TRANSACTIONS CLEAN",
        df
    )

    print(
        f"[TRANSACTIONS] rejected rows: "
        f"{len(rejects):,}"
    )

    return df


# ================================================================
# 7. SCHEME PERFORMANCE CLEANING
# ================================================================

def clean_scheme_performance():

    print()
    print("=" * 70)
    print("3. CLEANING 07_scheme_performance.csv")
    print("=" * 70)

    df = read_csv(
        "07_scheme_performance.csv"
    )

    print_report(
        "PERFORMANCE RAW",
        df
    )

    col_amfi = find_column(
        df,
        [
            "amfi_code",
            "scheme_code",
            "amficode",
            "code"
        ],
        label="amfi_code"
    )

    col_1y = find_column(
        df,
        [
            "return_1y",
            "return_1yr",
            "return_1yr_pct",
            "1y_return",
            "1yr_return",
            "return1y"
        ],
        required=False,
        label="return_1y"
    )

    col_3y = find_column(
        df,
        [
            "return_3y",
            "return_3yr",
            "return_3yr_pct",
            "3y_return",
            "3yr_return",
            "return3y"
        ],
        required=False,
        label="return_3y"
    )

    col_5y = find_column(
        df,
        [
            "return_5y",
            "return_5yr",
            "return_5yr_pct",
            "5y_return",
            "5yr_return",
            "return5y"
        ],
        required=False,
        label="return_5y"
    )

    col_expense = find_column(
        df,
        [
            "expense_ratio",
            "expense_ratio_pct",
            "expenseratio"
        ],
        label="expense_ratio"
    )

    col_aum = find_column(
        df,
        [
            "aum_crore",
            "aum",
            "assets_under_management",
            "aum_in_crore"
        ],
        required=False,
        label="aum_crore"
    )

    col_date = find_column(
        df,
        [
            "date",
            "as_of_date",
            "performance_date"
        ],
        required=False,
        label="date"
    )

    rename = {
        col_amfi: "amfi_code",
        col_expense: "expense_ratio"
    }

    if col_1y:
        rename[col_1y] = "return_1y"

    if col_3y:
        rename[col_3y] = "return_3y"

    if col_5y:
        rename[col_5y] = "return_5y"

    if col_aum:
        rename[col_aum] = "aum_crore"

    if col_date:
        rename[col_date] = "date"

    df = df.rename(
        columns=rename
    )

    df["amfi_code"] = clean_code_series(
        df["amfi_code"]
    )

    # ------------------------------------------------------------
    # Date
    # ------------------------------------------------------------

    if "date" in df.columns:

        df["date"] = parse_dates(
            df["date"]
        )

    else:

        # No date transformation is specified in the Day-2
        # description for this dataset. Use a nullable date.
        df["date"] = pd.NaT

    # ------------------------------------------------------------
    # Return columns
    # ------------------------------------------------------------

    return_columns = [
        column
        for column in [
            "return_1y",
            "return_3y",
            "return_5y"
        ]
        if column in df.columns
    ]

    for column in return_columns:

        df[column] = pd.to_numeric(
            df[column],
            errors="coerce"
        )

    # ------------------------------------------------------------
    # Expense ratio
    # ------------------------------------------------------------

    df["expense_ratio"] = pd.to_numeric(
        df["expense_ratio"],
        errors="coerce"
    )

    # ------------------------------------------------------------
    # AUM
    # ------------------------------------------------------------

    if "aum_crore" in df.columns:

        df["aum_crore"] = pd.to_numeric(
            df["aum_crore"],
            errors="coerce"
        )

    else:

        df["aum_crore"] = pd.NA

    # ------------------------------------------------------------
    # Anomaly flags
    #
    # The Day-2 description says to validate numeric returns and
    # flag anomalies. It does not specify a return percentage
    # threshold, so we do not invent one.
    # ------------------------------------------------------------

    anomaly_flags = pd.DataFrame(
        index=df.index
    )

    for column in return_columns:

        anomaly_flags[
            f"{column}_invalid_numeric"
        ] = df[column].isna()

    anomaly_flags[
        "expense_ratio_invalid"
    ] = (
        df["expense_ratio"].isna()
        |
        ~df["expense_ratio"].between(
            0.1,
            2.5,
            inclusive="both"
        )
    )

    if "date" in df.columns:

        anomaly_flags[
            "date_invalid"
        ] = df["date"].isna()

    df["is_anomaly"] = (
        anomaly_flags.any(
            axis=1
        )
    )

    df["anomaly_reasons"] = (
        anomaly_flags.apply(
            lambda row:
            ",".join(
                row.index[
                    row.astype(bool)
                ]
            ),
            axis=1
        )
    )

    # ------------------------------------------------------------
    # Keep the cleaned dataset.
    #
    # Anomalies are FLAGGED rather than silently removed because
    # the Day-2 requirement explicitly says to flag anomalies.
    # ------------------------------------------------------------

    write_csv(
        df,
        "07_scheme_performance_clean.csv"
    )

    anomalies = df[
        df["is_anomaly"]
    ].copy()

    if not anomalies.empty:

        write_csv(
            anomalies,
            "07_scheme_performance_anomalies.csv"
        )

    print_report(
        "PERFORMANCE CLEAN",
        df
    )

    print(
        f"[PERFORMANCE] flagged anomalies: "
        f"{len(anomalies):,}"
    )

    return df


# ================================================================
# 8. COPY DATASETS WITHOUT UNSPECIFIED TRANSFORMATIONS
# ================================================================

def copy_unchanged_datasets():

    print()
    print("=" * 70)
    print("4. COPYING DATASETS WITHOUT UNSPECIFIED TRANSFORMATIONS")
    print("=" * 70)

    copy_map = {
        "01_fund_master.csv":
            "01_fund_master.csv",

        "03_aum_by_fund_house.csv":
            "03_aum_by_fund_house.csv",

        "04_monthly_sip_inflows.csv":
            "04_monthly_sip_inflows.csv",

        "05_category_inflows.csv":
            "05_category_inflows.csv",

        "06_industry_folio_count.csv":
            "06_industry_folio_count.csv",

        "09_portfolio_holdings.csv":
            "09_portfolio_holdings.csv",

        "10_benchmark_indices.csv":
            "10_benchmark_indices.csv",
    }

    for source, destination in copy_map.items():

        source_path = RAW / source
        destination_path = (
            PROCESSED / destination
        )

        shutil.copy2(
            source_path,
            destination_path
        )

        print(
            f"[COPY] {source}"
            f" -> {destination}"
        )


# ================================================================
# 9. SQLITE SCHEMA
# ================================================================

SCHEMA_SQL = """
PRAGMA foreign_keys = ON;

CREATE TABLE IF NOT EXISTS dim_fund (
    amfi_code       TEXT PRIMARY KEY,
    scheme_name     TEXT NOT NULL,
    fund_house      TEXT,
    scheme_category TEXT,
    scheme_type     TEXT,
    risk_grade      TEXT,
    is_active       INTEGER DEFAULT 1
);

CREATE TABLE IF NOT EXISTS dim_date (
    date_id      TEXT PRIMARY KEY,
    year         INTEGER NOT NULL,
    quarter      INTEGER NOT NULL,
    month        INTEGER NOT NULL,
    month_name   TEXT NOT NULL,
    day          INTEGER NOT NULL,
    day_of_week  TEXT NOT NULL,
    is_weekend   INTEGER NOT NULL,
    is_holiday   INTEGER DEFAULT 0
);

CREATE TABLE IF NOT EXISTS dim_investor (
    investor_id TEXT PRIMARY KEY,
    state       TEXT,
    kyc_status  TEXT
);

CREATE TABLE IF NOT EXISTS fact_nav (
    nav_id      INTEGER PRIMARY KEY AUTOINCREMENT,
    amfi_code   TEXT NOT NULL,
    date_id     TEXT NOT NULL,
    nav         REAL NOT NULL CHECK (nav > 0),

    FOREIGN KEY (amfi_code)
        REFERENCES dim_fund(amfi_code),

    FOREIGN KEY (date_id)
        REFERENCES dim_date(date_id),

    UNIQUE(amfi_code, date_id)
);

CREATE TABLE IF NOT EXISTS fact_transactions (
    transaction_id   TEXT PRIMARY KEY,
    investor_id      TEXT NOT NULL,
    amfi_code        TEXT NOT NULL,
    date_id          TEXT NOT NULL,

    transaction_type TEXT NOT NULL
        CHECK (
            transaction_type
            IN ('SIP', 'Lumpsum', 'Redemption')
        ),

    amount REAL NOT NULL CHECK (amount > 0),

    FOREIGN KEY (investor_id)
        REFERENCES dim_investor(investor_id),

    FOREIGN KEY (amfi_code)
        REFERENCES dim_fund(amfi_code),

    FOREIGN KEY (date_id)
        REFERENCES dim_date(date_id)
);

CREATE TABLE IF NOT EXISTS fact_performance (
    performance_id INTEGER PRIMARY KEY AUTOINCREMENT,

    amfi_code      TEXT NOT NULL,
    date_id        TEXT,

    return_1y      REAL,
    return_3y      REAL,
    return_5y      REAL,

    expense_ratio REAL
        CHECK (
            expense_ratio
            BETWEEN 0.1 AND 2.5
        ),

    aum_crore REAL,

    is_anomaly INTEGER DEFAULT 0,

    FOREIGN KEY (amfi_code)
        REFERENCES dim_fund(amfi_code),

    FOREIGN KEY (date_id)
        REFERENCES dim_date(date_id),

    UNIQUE(amfi_code, date_id)
);

CREATE TABLE IF NOT EXISTS fact_aum (
    aum_id      INTEGER PRIMARY KEY AUTOINCREMENT,

    amfi_code   TEXT,
    fund_house  TEXT NOT NULL,
    date_id     TEXT NOT NULL,

    aum_crore   REAL,
    aum_lakh_crore REAL,
    num_schemes INTEGER,

    FOREIGN KEY (amfi_code)
        REFERENCES dim_fund(amfi_code),

    FOREIGN KEY (date_id)
        REFERENCES dim_date(date_id)
);

CREATE INDEX IF NOT EXISTS idx_nav_amfi_date
ON fact_nav(amfi_code, date_id);

CREATE INDEX IF NOT EXISTS idx_transaction_amfi_date
ON fact_transactions(amfi_code, date_id);

CREATE INDEX IF NOT EXISTS idx_transaction_investor
ON fact_transactions(investor_id);

CREATE INDEX IF NOT EXISTS idx_performance_amfi_date
ON fact_performance(amfi_code, date_id);

CREATE INDEX IF NOT EXISTS idx_aum_fund_house_date
ON fact_aum(fund_house, date_id);
"""


# ================================================================
# 10. BUILD DATE DIMENSION
# ================================================================

def build_dim_date(date_series):

    dates = (
        pd.to_datetime(
            pd.Series(date_series),
            errors="coerce"
        )
        .dropna()
        .dt.normalize()
        .drop_duplicates()
        .sort_values()
    )

    dim = pd.DataFrame(
        {
            "date_id":
            dates.dt.strftime(
                "%Y-%m-%d"
            )
        }
    )

    parsed = pd.to_datetime(
        dim["date_id"]
    )

    dim["year"] = parsed.dt.year
    dim["quarter"] = parsed.dt.quarter
    dim["month"] = parsed.dt.month
    dim["month_name"] = (
        parsed.dt.month_name()
    )
    dim["day"] = parsed.dt.day
    dim["day_of_week"] = (
        parsed.dt.day_name()
    )

    dim["is_weekend"] = (
        parsed.dt.dayofweek
        .isin([5, 6])
        .astype(int)
    )

    # No holiday calendar was supplied in the Day-2 datasets.
    dim["is_holiday"] = 0

    return dim


# ================================================================
# 11. CREATE / RESET DATABASE
# ================================================================

def create_database():

    print()
    print("=" * 70)
    print("5. CREATING SQLITE DATABASE")
    print("=" * 70)

    # ------------------------------------------------------------
    # Dispose previous SQLAlchemy engine if this script was
    # previously executed in the same Jupyter kernel.
    # ------------------------------------------------------------

    old_engine = globals().get(
        "engine"
    )

    if old_engine is not None:

        try:

            old_engine.dispose()

            print(
                "[DATABASE] Previous engine disposed."
            )

        except Exception as error:

            print(
                "[DATABASE] Engine dispose warning:",
                error
            )

    # ------------------------------------------------------------
    # IMPORTANT:
    #
    # Do NOT delete bluestock_mf.db.
    #
    # Windows may have the SQLite file open through VS Code,
    # Jupyter, SQLite Viewer, etc.
    #
    # Instead, reuse the database and drop the Day-2 tables.
    # ------------------------------------------------------------

    connection = sqlite3.connect(
        DB_PATH,
        timeout=30
    )

    connection.execute(
        "PRAGMA busy_timeout = 30000"
    )

    connection.execute(
        "PRAGMA foreign_keys = OFF"
    )

    tables = [
        "fact_aum",
        "fact_performance",
        "fact_transactions",
        "fact_nav",
        "dim_investor",
        "dim_date",
        "dim_fund"
    ]

    for table in tables:

        connection.execute(
            f"DROP TABLE IF EXISTS {table}"
        )

    connection.commit()

    connection.executescript(
        SCHEMA_SQL
    )

    connection.commit()
    connection.close()

    print(
        f"[DATABASE] Ready: {DB_PATH}"
    )

    # ------------------------------------------------------------
    # Write schema.sql
    # ------------------------------------------------------------

    SCHEMA_PATH.write_text(
        SCHEMA_SQL.strip()
        + "\n",
        encoding="utf-8"
    )

    print(
        f"[WRITE] {SCHEMA_PATH}"
    )

    # ------------------------------------------------------------
    # SQLAlchemy
    # ------------------------------------------------------------

    engine = create_engine(
        f"sqlite:///{DB_PATH}",
        connect_args={
            "timeout": 30
        }
    )

    return engine


# ================================================================
# 12. LOAD DIM FUND
# ================================================================

def load_dim_fund(
    engine,
    nav,
    transactions,
    performance
):

    df = pd.read_csv(
        PROCESSED /
        "01_fund_master.csv"
    )

    print_report(
        "FUND MASTER",
        df
    )

    col_amfi = find_column(
        df,
        [
            "amfi_code",
            "scheme_code",
            "amficode",
            "code"
        ],
        label="amfi_code"
    )

    col_name = find_column(
        df,
        [
            "scheme_name",
            "fund_name",
            "name"
        ],
        required=False,
        label="scheme_name"
    )

    col_house = find_column(
        df,
        [
            "fund_house",
            "amc",
            "amc_name"
        ],
        required=False,
        label="fund_house"
    )

    col_category = find_column(
        df,
        [
            "category",
            "scheme_category"
        ],
        required=False,
        label="scheme_category"
    )

    col_type = find_column(
        df,
        [
            "sub_category",
            "subcategory",
            "scheme_type"
        ],
        required=False,
        label="scheme_type"
    )

    col_risk = find_column(
        df,
        [
            "risk_category",
            "risk_grade",
            "risk"
        ],
        required=False,
        label="risk_grade"
    )

    rename = {
        col_amfi: "amfi_code"
    }

    if col_name:
        rename[col_name] = "scheme_name"

    if col_house:
        rename[col_house] = "fund_house"

    if col_category:
        rename[col_category] = "scheme_category"

    if col_type:
        rename[col_type] = "scheme_type"

    if col_risk:
        rename[col_risk] = "risk_grade"

    df = df.rename(
        columns=rename
    )

    if "scheme_name" not in df.columns:
        df["scheme_name"] = df["amfi_code"]

    if "fund_house" not in df.columns:
        df["fund_house"] = None

    if "scheme_category" not in df.columns:
        df["scheme_category"] = None

    if "scheme_type" not in df.columns:
        df["scheme_type"] = None

    if "risk_grade" not in df.columns:
        df["risk_grade"] = None

    df["amfi_code"] = clean_code_series(
        df["amfi_code"]
    )

    df["scheme_name"] = (
        df["scheme_name"]
        .fillna(df["amfi_code"])
        .astype(str)
    )

    df["is_active"] = 1

    df = df[
        [
            "amfi_code",
            "scheme_name",
            "fund_house",
            "scheme_category",
            "scheme_type",
            "risk_grade",
            "is_active"
        ]
    ]

    # ------------------------------------------------------------
    # Add codes that occur in facts but are absent from fund master.
    #
    # This prevents foreign-key failure without inventing
    # additional descriptive attributes.
    # ------------------------------------------------------------

    known_codes = set(
        df["amfi_code"]
    )

    fact_codes = (
        set(nav["amfi_code"])
        |
        set(transactions["amfi_code"])
        |
        set(performance["amfi_code"])
    )

    missing_codes = (
        fact_codes - known_codes
    )

    if missing_codes:

        extra = pd.DataFrame(
            {
                "amfi_code":
                sorted(missing_codes)
            }
        )

        extra["scheme_name"] = (
            extra["amfi_code"]
        )

        extra["fund_house"] = None
        extra["scheme_category"] = None
        extra["scheme_type"] = None
        extra["risk_grade"] = None
        extra["is_active"] = 1

        df = pd.concat(
            [
                df,
                extra
            ],
            ignore_index=True
        )

    df = df.drop_duplicates(
        subset=["amfi_code"],
        keep="first"
    )

    df.to_sql(
        "dim_fund",
        engine,
        if_exists="append",
        index=False
    )

    print(
        f"[dim_fund] loaded {len(df):,} rows"
    )


# ================================================================
# 13. LOAD DIM DATE
# ================================================================

def load_dim_date(
    engine,
    nav,
    transactions,
    performance,
    aum
):

    dates = pd.concat(
        [
            nav["date"],
            transactions["date"],
            performance["date"],
            aum["date"]
        ],
        ignore_index=True
    )

    dim_date = build_dim_date(
        dates
    )

    dim_date.to_sql(
        "dim_date",
        engine,
        if_exists="append",
        index=False
    )

    print(
        f"[dim_date] loaded {len(dim_date):,} rows"
    )


# ================================================================
# 14. LOAD DIM INVESTOR
# ================================================================

def load_dim_investor(
    engine,
    transactions
):

    columns = [
        "investor_id",
        "kyc_status"
    ]

    if "state" in transactions.columns:
        columns.append("state")

    df = transactions[
        columns
    ].copy()

    if "state" not in df.columns:
        df["state"] = None

    df = df[
        [
            "investor_id",
            "state",
            "kyc_status"
        ]
    ]

    df = df.drop_duplicates(
        subset=["investor_id"],
        keep="last"
    )

    df.to_sql(
        "dim_investor",
        engine,
        if_exists="append",
        index=False
    )

    print(
        f"[dim_investor] loaded "
        f"{len(df):,} rows"
    )


# ================================================================
# 15. LOAD FACT NAV
# ================================================================

def load_nav_fact(
    engine,
    nav
):

    df = nav.copy()

    df["date_id"] = (
        df["date"]
        .dt.strftime(
            "%Y-%m-%d"
        )
    )

    df = df[
        [
            "amfi_code",
            "date_id",
            "nav"
        ]
    ]

    df = df.drop_duplicates(
        subset=[
            "amfi_code",
            "date_id"
        ],
        keep="last"
    )

    df.to_sql(
        "fact_nav",
        engine,
        if_exists="append",
        index=False
    )

    print(
        f"[fact_nav] loaded {len(df):,} rows"
    )

    return len(df)


# ================================================================
# 16. LOAD FACT TRANSACTIONS
# ================================================================

def load_transaction_fact(
    engine,
    transactions
):

    df = transactions.copy()

    df["date_id"] = (
        df["date"]
        .dt.strftime(
            "%Y-%m-%d"
        )
    )

    df = df[
        [
            "transaction_id",
            "investor_id",
            "amfi_code",
            "date_id",
            "transaction_type",
            "amount"
        ]
    ]

    df.to_sql(
        "fact_transactions",
        engine,
        if_exists="append",
        index=False
    )

    print(
        f"[fact_transactions] "
        f"loaded {len(df):,} rows"
    )

    return len(df)


# ================================================================
# 17. LOAD FACT PERFORMANCE
# ================================================================

def load_performance_fact(
    engine,
    performance
):

    df = performance.copy()

    # ------------------------------------------------------------
    # Required optional columns
    # ------------------------------------------------------------

    for column in [
        "return_1y",
        "return_3y",
        "return_5y",
        "aum_crore"
    ]:

        if column not in df.columns:

            df[column] = None

    # ------------------------------------------------------------
    # Date
    # ------------------------------------------------------------

    df["date_id"] = (
        df["date"]
        .dt.strftime(
            "%Y-%m-%d"
        )
    )

    # ------------------------------------------------------------
    # SQLite UNIQUE(amfi_code, date_id) does not prevent multiple
    # NULL dates, so duplicate AMFI/date rows are handled here.
    # ------------------------------------------------------------

    df = df.drop_duplicates(
        subset=[
            "amfi_code",
            "date_id"
        ],
        keep="last"
    )

    # ------------------------------------------------------------
    # Store anomaly status
    # ------------------------------------------------------------

    if "is_anomaly" in df.columns:

        df["is_anomaly"] = (
            df["is_anomaly"]
            .fillna(False)
            .astype(int)
        )

    else:

        df["is_anomaly"] = 0

    df = df[
        [
            "amfi_code",
            "date_id",
            "return_1y",
            "return_3y",
            "return_5y",
            "expense_ratio",
            "aum_crore",
            "is_anomaly"
        ]
    ]

    df.to_sql(
        "fact_performance",
        engine,
        if_exists="append",
        index=False
    )

    print(
        f"[fact_performance] "
        f"loaded {len(df):,} rows"
    )

    return len(df)


# ================================================================
# 18. LOAD FACT AUM
# ================================================================

def load_aum_fact(
    engine
):

    print()
    print("=" * 70)
    print("LOADING FACT_AUM")
    print("=" * 70)

    # ------------------------------------------------------------
    # IMPORTANT:
    #
    # Actual 03_aum_by_fund_house.csv columns from your dataset:
    #
    # date
    # fund_house
    # aum_lakh_crore
    # aum_crore
    # num_schemes
    #
    # There is NO year column.
    # ------------------------------------------------------------

    df = pd.read_csv(
        PROCESSED /
        "03_aum_by_fund_house.csv"
    )

    print_report(
        "AUM RAW",
        df
    )

    col_date = find_column(
        df,
        [
            "date"
        ],
        label="date"
    )

    col_house = find_column(
        df,
        [
            "fund_house",
            "fundhouse",
            "amc",
            "amc_name"
        ],
        label="fund_house"
    )

    col_lakh = find_column(
        df,
        [
            "aum_lakh_crore"
        ],
        required=False,
        label="aum_lakh_crore"
    )

    col_crore = find_column(
        df,
        [
            "aum_crore",
            "aum"
        ],
        required=False,
        label="aum_crore"
    )

    col_schemes = find_column(
        df,
        [
            "num_schemes",
            "number_of_schemes",
            "scheme_count"
        ],
        required=False,
        label="num_schemes"
    )

    rename = {
        col_date: "date",
        col_house: "fund_house"
    }

    if col_lakh:
        rename[col_lakh] = "aum_lakh_crore"

    if col_crore:
        rename[col_crore] = "aum_crore"

    if col_schemes:
        rename[col_schemes] = "num_schemes"

    df = df.rename(
        columns=rename
    )

    if "aum_lakh_crore" not in df.columns:
        df["aum_lakh_crore"] = None

    if "aum_crore" not in df.columns:
        df["aum_crore"] = None

    if "num_schemes" not in df.columns:
        df["num_schemes"] = None

    df["date"] = parse_dates(
        df["date"]
    )

    df["aum_lakh_crore"] = pd.to_numeric(
        df["aum_lakh_crore"],
        errors="coerce"
    )

    df["aum_crore"] = pd.to_numeric(
        df["aum_crore"],
        errors="coerce"
    )

    df["num_schemes"] = pd.to_numeric(
        df["num_schemes"],
        errors="coerce"
    )

    df["date_id"] = (
        df["date"]
        .dt.strftime(
            "%Y-%m-%d"
        )
    )

    # AUM is fund-house-level in this dataset.
    # Therefore amfi_code is intentionally NULL.

    df["amfi_code"] = None

    df = df[
        [
            "amfi_code",
            "fund_house",
            "date_id",
            "aum_crore",
            "aum_lakh_crore",
            "num_schemes"
        ]
    ]

    df = df.dropna(
        subset=[
            "date_id",
            "fund_house"
        ]
    )

    df.to_sql(
        "fact_aum",
        engine,
        if_exists="append",
        index=False
    )

    print(
        f"[fact_aum] loaded {len(df):,} rows"
    )

    return len(df)


# ================================================================
# 19. ANALYTICAL QUERIES
# ================================================================

QUERIES_SQL = """

-- ============================================================
-- BLUESTOCK MF - DAY 2 ANALYTICAL QUERIES
-- ============================================================


-- 1. Top 5 funds by scheme-level AUM
-- AUM comes from 07_scheme_performance.csv.

SELECT
    f.amfi_code,
    f.scheme_name,
    p.aum_crore
FROM fact_performance p
JOIN dim_fund f
    ON p.amfi_code = f.amfi_code
WHERE p.aum_crore IS NOT NULL
ORDER BY p.aum_crore DESC
LIMIT 5;


-- ============================================================
-- 2. Average NAV per month
-- ============================================================

SELECT
    substr(n.date_id, 1, 7) AS month,
    ROUND(AVG(n.nav), 4) AS average_nav
FROM fact_nav n
GROUP BY substr(n.date_id, 1, 7)
ORDER BY month;


-- ============================================================
-- 3. SIP year-over-year growth
-- ============================================================

WITH yearly_sip AS (

    SELECT
        d.year,
        SUM(t.amount) AS sip_amount
    FROM fact_transactions t
    JOIN dim_date d
        ON t.date_id = d.date_id
    WHERE t.transaction_type = 'SIP'
    GROUP BY d.year
),

growth AS (

    SELECT
        year,
        sip_amount,
        LAG(sip_amount)
            OVER (
                ORDER BY year
            ) AS previous_year_sip
    FROM yearly_sip
)

SELECT
    year,
    ROUND(sip_amount, 2) AS sip_amount,
    ROUND(previous_year_sip, 2)
        AS previous_year_sip,

    ROUND(
        CASE
            WHEN previous_year_sip IS NULL
                 OR previous_year_sip = 0
            THEN NULL

            ELSE
                (
                    (sip_amount - previous_year_sip)
                    / previous_year_sip
                ) * 100
        END,
        2
    ) AS yoy_growth_pct

FROM growth
ORDER BY year;


-- ============================================================
-- 4. Transactions by state
-- ============================================================

SELECT
    i.state,
    COUNT(t.transaction_id)
        AS transaction_count,
    ROUND(
        SUM(t.amount),
        2
    ) AS total_amount
FROM fact_transactions t
JOIN dim_investor i
    ON t.investor_id = i.investor_id
GROUP BY i.state
ORDER BY total_amount DESC;


-- ============================================================
-- 5. Funds with expense ratio below 1%
-- ============================================================

SELECT
    f.amfi_code,
    f.scheme_name,
    p.expense_ratio
FROM fact_performance p
JOIN dim_fund f
    ON p.amfi_code = f.amfi_code
WHERE p.expense_ratio < 1.0
ORDER BY p.expense_ratio ASC;


-- ============================================================
-- 6. Top 5 funds by transaction amount
-- ============================================================

SELECT
    f.amfi_code,
    f.scheme_name,
    COUNT(t.transaction_id)
        AS transaction_count,
    ROUND(
        SUM(t.amount),
        2
    ) AS total_transaction_amount
FROM fact_transactions t
JOIN dim_fund f
    ON t.amfi_code = f.amfi_code
GROUP BY
    f.amfi_code,
    f.scheme_name
ORDER BY total_transaction_amount DESC
LIMIT 5;


-- ============================================================
-- 7. Latest NAV for each fund
-- ============================================================

SELECT
    f.amfi_code,
    f.scheme_name,
    n.date_id,
    n.nav
FROM fact_nav n
JOIN dim_fund f
    ON n.amfi_code = f.amfi_code
WHERE n.date_id = (
    SELECT MAX(n2.date_id)
    FROM fact_nav n2
    WHERE n2.amfi_code = n.amfi_code
)
ORDER BY n.nav DESC;


-- ============================================================
-- 8. Fund performance and expense ratio
-- ============================================================

SELECT
    f.amfi_code,
    f.scheme_name,
    p.return_1y,
    p.return_3y,
    p.return_5y,
    p.expense_ratio
FROM fact_performance p
JOIN dim_fund f
    ON p.amfi_code = f.amfi_code
ORDER BY p.return_1y DESC;


-- ============================================================
-- 9. Fund-house AUM
-- ============================================================

SELECT
    fund_house,
    ROUND(
        SUM(aum_crore),
        2
    ) AS total_aum_crore,
    SUM(num_schemes)
        AS total_schemes
FROM fact_aum
GROUP BY fund_house
ORDER BY total_aum_crore DESC;


-- ============================================================
-- 10. Transaction count and amount by KYC status
-- ============================================================

SELECT
    i.kyc_status,
    COUNT(t.transaction_id)
        AS transaction_count,
    ROUND(
        SUM(t.amount),
        2
    ) AS total_amount
FROM fact_transactions t
JOIN dim_investor i
    ON t.investor_id = i.investor_id
GROUP BY i.kyc_status
ORDER BY total_amount DESC;

"""


# ================================================================
# 20. WRITE QUERIES.SQL
# ================================================================

def write_queries_file():

    QUERIES_PATH.write_text(
        QUERIES_SQL.strip()
        + "\n",
        encoding="utf-8"
    )

    print(
        f"[WRITE] {QUERIES_PATH}"
    )


# ================================================================
# 21. DATA DICTIONARY
# ================================================================

DATA_DICTIONARY = """
# Bluestock MF — Day 2 Data Dictionary

## Project

Bluestock Mutual Fund analytics pipeline.

The Day-2 pipeline cleans selected source datasets, creates a SQLite
star schema, loads cleaned data, and provides analytical SQL queries.

---

# Source Datasets

## 01_fund_master.csv

Master information about mutual fund schemes.

| Column | Data Type | Business Definition | Source |
|---|---|---|---|
| amfi_code | TEXT | AMFI identifier for the scheme | 01_fund_master.csv |
| scheme_name | TEXT | Name of the mutual fund scheme | 01_fund_master.csv |
| fund_house | TEXT | Asset management company/fund house | 01_fund_master.csv |
| category / scheme_category | TEXT | Broad scheme category | 01_fund_master.csv |
| sub_category / scheme_type | TEXT | Scheme sub-category/type | 01_fund_master.csv |
| risk_category / risk_grade | TEXT | Risk classification where supplied | 01_fund_master.csv |

---

## 02_nav_history.csv

Historical NAV observations.

| Column | Data Type | Business Definition | Source |
|---|---|---|---|
| amfi_code | TEXT | AMFI scheme identifier | 02_nav_history.csv |
| date | DATE | NAV observation date | 02_nav_history.csv |
| nav | REAL | Net asset value of the scheme | 02_nav_history.csv |

Day-2 transformations:

- Dates parsed to datetime.
- Records sorted by AMFI code and date.
- Duplicate AMFI/date observations removed.
- NAV validated as greater than zero.
- Missing NAV values are forward-filled within the same AMFI code where a previous valid NAV exists.

---

## 03_aum_by_fund_house.csv

Fund-house-level AUM data.

| Column | Data Type | Business Definition | Source |
|---|---|---|---|
| date | DATE | Reporting date | 03_aum_by_fund_house.csv |
| fund_house | TEXT | Mutual fund house/AMC | 03_aum_by_fund_house.csv |
| aum_lakh_crore | REAL | AUM expressed in lakh crore units | 03_aum_by_fund_house.csv |
| aum_crore | REAL | AUM expressed in crore units | 03_aum_by_fund_house.csv |
| num_schemes | INTEGER/REAL | Number of schemes reported for the fund house | 03_aum_by_fund_house.csv |

This dataset is treated as fund-house-level AUM.
It is not treated as scheme-level AUM.

---

## 04_monthly_sip_inflows.csv

Monthly SIP inflow data.

No Day-2 transformation rule was specified, so the dataset is copied
to data/processed without modification.

---

## 05_category_inflows.csv

Category-level inflow data.

No Day-2 transformation rule was specified, so the dataset is copied
to data/processed without modification.

---

## 06_industry_folio_count.csv

Industry folio-count data.

No Day-2 transformation rule was specified, so the dataset is copied
to data/processed without modification.

---

## 07_scheme_performance.csv

Scheme-level performance information.

| Column | Data Type | Business Definition | Source |
|---|---|---|---|
| amfi_code | TEXT | AMFI scheme identifier | 07_scheme_performance.csv |
| return_1y | REAL | One-year scheme return | 07_scheme_performance.csv |
| return_3y | REAL | Three-year scheme return | 07_scheme_performance.csv |
| return_5y | REAL | Five-year scheme return | 07_scheme_performance.csv |
| expense_ratio | REAL | Scheme expense ratio | 07_scheme_performance.csv |
| aum_crore | REAL | Scheme-level AUM in crore, where supplied | 07_scheme_performance.csv |

Day-2 transformations:

- Return fields converted to numeric values.
- Invalid/non-numeric return values are flagged.
- Expense ratio converted to numeric.
- Expense ratio is checked against the specified 0.1%–2.5% range.
- Anomaly status and anomaly reasons are retained in the cleaned output.
- A separate anomaly CSV is also produced.

No additional return anomaly threshold is invented because the Day-2
description only requires numeric validation and anomaly flagging.

---

## 08_investor_transactions.csv

Investor transaction records.

| Column | Data Type | Business Definition | Source |
|---|---|---|---|
| transaction_id | TEXT | Unique transaction identifier | 08_investor_transactions.csv |
| investor_id | TEXT | Investor identifier | 08_investor_transactions.csv |
| amfi_code | TEXT | AMFI scheme identifier | 08_investor_transactions.csv |
| date | DATE | Transaction date | 08_investor_transactions.csv |
| transaction_type | TEXT | SIP, Lumpsum, or Redemption | 08_investor_transactions.csv |
| amount | REAL | Transaction amount | 08_investor_transactions.csv |
| kyc_status | TEXT | Investor KYC status | 08_investor_transactions.csv |
| state | TEXT | Investor state where supplied | 08_investor_transactions.csv |

Day-2 transformations:

- Transaction types standardised to SIP, Lumpsum, and Redemption.
- Amount validated as greater than zero.
- Dates parsed to datetime.
- KYC status checked against the supported enum values.
- Duplicate transaction IDs removed.

---

## 09_portfolio_holdings.csv

Portfolio holding information.

No Day-2 transformation rule was specified, so the dataset is copied
to data/processed without modification.

---

## 10_benchmark_indices.csv

Benchmark index information.

No Day-2 transformation rule was specified, so the dataset is copied
to data/processed without modification.

---

# SQLite Star Schema

## dim_fund

| Column | Type | Definition |
|---|---|---|
| amfi_code | TEXT PK | Unique AMFI scheme identifier |
| scheme_name | TEXT | Scheme name |
| fund_house | TEXT | Fund house |
| scheme_category | TEXT | Scheme category |
| scheme_type | TEXT | Scheme type/sub-category |
| risk_grade | TEXT | Risk classification |
| is_active | INTEGER | Active flag |

---

## dim_date

| Column | Type | Definition |
|---|---|---|
| date_id | TEXT PK | Date in YYYY-MM-DD format |
| year | INTEGER | Calendar year |
| quarter | INTEGER | Calendar quarter |
| month | INTEGER | Calendar month |
| month_name | TEXT | Month name |
| day | INTEGER | Day of month |
| day_of_week | TEXT | Day name |
| is_weekend | INTEGER | Weekend flag |
| is_holiday | INTEGER | Holiday flag; no holiday calendar supplied |

---

## dim_investor

| Column | Type | Definition |
|---|---|---|
| investor_id | TEXT PK | Unique investor identifier |
| state | TEXT | Investor state |
| kyc_status | TEXT | KYC status |

---

## fact_nav

| Column | Type | Definition |
|---|---|---|
| nav_id | INTEGER PK | Surrogate NAV record ID |
| amfi_code | TEXT FK | Fund identifier |
| date_id | TEXT FK | Observation date |
| nav | REAL | NAV value; must be greater than zero |

---

## fact_transactions

| Column | Type | Definition |
|---|---|---|
| transaction_id | TEXT PK | Transaction identifier |
| investor_id | TEXT FK | Investor identifier |
| amfi_code | TEXT FK | Fund identifier |
| date_id | TEXT FK | Transaction date |
| transaction_type | TEXT | SIP, Lumpsum, or Redemption |
| amount | REAL | Transaction amount |

---

## fact_performance

| Column | Type | Definition |
|---|---|---|
| performance_id | INTEGER PK | Surrogate performance record ID |
| amfi_code | TEXT FK | Fund identifier |
| date_id | TEXT FK | Performance date where supplied |
| return_1y | REAL | One-year return |
| return_3y | REAL | Three-year return |
| return_5y | REAL | Five-year return |
| expense_ratio | REAL | Expense ratio |
| aum_crore | REAL | Scheme-level AUM |
| is_anomaly | INTEGER | Anomaly flag |

---

## fact_aum

| Column | Type | Definition |
|---|---|---|
| aum_id | INTEGER PK | Surrogate AUM record ID |
| amfi_code | TEXT FK/NULL | Scheme identifier when applicable |
| fund_house | TEXT | Fund-house identifier/name |
| date_id | TEXT FK | AUM reporting date |
| aum_crore | REAL | AUM in crore |
| aum_lakh_crore | REAL | AUM in lakh crore |
| num_schemes | INTEGER/REAL | Number of schemes |

The current 03_aum_by_fund_house.csv dataset is fund-house-level,
so amfi_code is intentionally NULL for those records.

---

# Deliverables

The pipeline creates:

- data/processed/01_fund_master.csv
- data/processed/02_nav_history_clean.csv
- data/processed/03_aum_by_fund_house.csv
- data/processed/04_monthly_sip_inflows.csv
- data/processed/05_category_inflows.csv
- data/processed/06_industry_folio_count.csv
- data/processed/07_scheme_performance_clean.csv
- data/processed/08_investor_transactions_clean.csv
- data/processed/09_portfolio_holdings.csv
- data/processed/10_benchmark_indices.csv
- data/processed/02_nav_history_rejects.csv where required
- data/processed/08_investor_transactions_rejects.csv where required
- data/processed/07_scheme_performance_anomalies.csv where required
- bluestock_mf.db
- schema.sql
- queries.sql
- data_dictionary.md
"""


# ================================================================
# 22. WRITE DATA DICTIONARY
# ================================================================

def write_data_dictionary():

    DICTIONARY_PATH.write_text(
        DATA_DICTIONARY.strip()
        + "\n",
        encoding="utf-8"
    )

    print(
        f"[WRITE] {DICTIONARY_PATH}"
    )


# ================================================================
# 23. VERIFY DATABASE
# ================================================================

def verify_database(
    engine,
    expected_counts
):

    print()
    print("=" * 70)
    print("DATABASE VERIFICATION")
    print("=" * 70)

    tables = [
        "dim_fund",
        "dim_date",
        "dim_investor",
        "fact_nav",
        "fact_transactions",
        "fact_performance",
        "fact_aum"
    ]

    with engine.connect() as connection:

        for table in tables:

            count = connection.exec_driver_sql(
                f"SELECT COUNT(*) FROM {table}"
            ).scalar()

            print(
                f"{table:25s}: {count:,}"
            )

        print()
        print(
            "FACT ROW COUNT VERIFICATION"
        )

        all_ok = True

        for table, expected in expected_counts.items():

            actual = connection.exec_driver_sql(
                f"SELECT COUNT(*) FROM {table}"
            ).scalar()

            status = (
                "OK"
                if actual == expected
                else "CHECK"
            )

            if actual != expected:
                all_ok = False

            print(
                f"{table:25s} "
                f"expected={expected:,} "
                f"actual={actual:,} "
                f"[{status}]"
            )

        print()

        # --------------------------------------------------------
        # Foreign keys
        # --------------------------------------------------------

        fk_errors = connection.exec_driver_sql(
            "PRAGMA foreign_key_check"
        ).fetchall()

        if fk_errors:

            print(
                f"[WARNING] "
                f"{len(fk_errors)} foreign-key errors."
            )

            for error in fk_errors[:10]:
                print(error)

        else:

            print(
                "[OK] Foreign-key integrity check passed."
            )

        if all_ok:

            print(
                "[OK] Fact row counts match expected cleaned data."
            )

        else:

            print(
                "[WARNING] One or more fact row counts differ."
            )


# ================================================================
# 24. RUN THE 10 QUERIES
# ================================================================

def run_queries(
    engine
):

    print()
    print("=" * 70)
    print("ANALYTICAL SQL QUERIES")
    print("=" * 70)

    queries = {

        "1. Top 5 funds by scheme-level AUM":
        """
        SELECT
            f.amfi_code,
            f.scheme_name,
            p.aum_crore
        FROM fact_performance p
        JOIN dim_fund f
            ON p.amfi_code = f.amfi_code
        WHERE p.aum_crore IS NOT NULL
        ORDER BY p.aum_crore DESC
        LIMIT 5;
        """,

        "2. Average NAV per month":
        """
        SELECT
            substr(date_id, 1, 7) AS month,
            ROUND(AVG(nav), 4) AS average_nav
        FROM fact_nav
        GROUP BY substr(date_id, 1, 7)
        ORDER BY month;
        """,

        "3. SIP YoY growth":
        """
        WITH yearly_sip AS (
            SELECT
                d.year,
                SUM(t.amount) AS sip_amount
            FROM fact_transactions t
            JOIN dim_date d
                ON t.date_id = d.date_id
            WHERE t.transaction_type = 'SIP'
            GROUP BY d.year
        ),

        growth AS (
            SELECT
                year,
                sip_amount,
                LAG(sip_amount)
                    OVER (
                        ORDER BY year
                    ) AS previous_year_sip
            FROM yearly_sip
        )

        SELECT
            year,
            ROUND(sip_amount, 2)
                AS sip_amount,
            ROUND(previous_year_sip, 2)
                AS previous_year_sip,
            ROUND(
                CASE
                    WHEN previous_year_sip IS NULL
                         OR previous_year_sip = 0
                    THEN NULL
                    ELSE
                        (
                            (sip_amount - previous_year_sip)
                            / previous_year_sip
                        ) * 100
                END,
                2
            ) AS yoy_growth_pct
        FROM growth
        ORDER BY year;
        """,

        "4. Transactions by state":
        """
        SELECT
            i.state,
            COUNT(t.transaction_id)
                AS transaction_count,
            ROUND(
                SUM(t.amount),
                2
            ) AS total_amount
        FROM fact_transactions t
        JOIN dim_investor i
            ON t.investor_id = i.investor_id
        GROUP BY i.state
        ORDER BY total_amount DESC;
        """,

        "5. Funds with expense ratio below 1%":
        """
        SELECT
            f.amfi_code,
            f.scheme_name,
            p.expense_ratio
        FROM fact_performance p
        JOIN dim_fund f
            ON p.amfi_code = f.amfi_code
        WHERE p.expense_ratio < 1.0
        ORDER BY p.expense_ratio ASC;
        """,

        "6. Top 5 funds by transaction amount":
        """
        SELECT
            f.amfi_code,
            f.scheme_name,
            COUNT(t.transaction_id)
                AS transaction_count,
            ROUND(
                SUM(t.amount),
                2
            ) AS total_transaction_amount
        FROM fact_transactions t
        JOIN dim_fund f
            ON t.amfi_code = f.amfi_code
        GROUP BY
            f.amfi_code,
            f.scheme_name
        ORDER BY total_transaction_amount DESC
        LIMIT 5;
        """,

        "7. Latest NAV for each fund":
        """
        SELECT
            f.amfi_code,
            f.scheme_name,
            n.date_id,
            n.nav
        FROM fact_nav n
        JOIN dim_fund f
            ON n.amfi_code = f.amfi_code
        WHERE n.date_id = (
            SELECT MAX(n2.date_id)
            FROM fact_nav n2
            WHERE n2.amfi_code = n.amfi_code
        )
        ORDER BY n.nav DESC;
        """,

        "8. Performance overview":
        """
        SELECT
            f.amfi_code,
            f.scheme_name,
            p.return_1y,
            p.return_3y,
            p.return_5y,
            p.expense_ratio
        FROM fact_performance p
        JOIN dim_fund f
            ON p.amfi_code = f.amfi_code
        ORDER BY p.return_1y DESC;
        """,

        "9. Fund-house AUM":
        """
        SELECT
            fund_house,
            ROUND(
                SUM(aum_crore),
                2
            ) AS total_aum_crore,
            SUM(num_schemes)
                AS total_schemes
        FROM fact_aum
        GROUP BY fund_house
        ORDER BY total_aum_crore DESC;
        """,

        "10. Transactions by KYC status":
        """
        SELECT
            i.kyc_status,
            COUNT(t.transaction_id)
                AS transaction_count,
            ROUND(
                SUM(t.amount),
                2
            ) AS total_amount
        FROM fact_transactions t
        JOIN dim_investor i
            ON t.investor_id = i.investor_id
        GROUP BY i.kyc_status
        ORDER BY total_amount DESC;
        """
    }

    for title, sql in queries.items():

        print()
        print("-" * 70)
        print(title)
        print("-" * 70)

        try:

            result = pd.read_sql(
                sql,
                engine
            )

            if result.empty:

                print(
                    "No rows returned."
                )

            else:

                print(
                    result.to_string(
                        index=False
                    )
                )

        except Exception as error:

            print(
                "[QUERY ERROR]",
                error
            )


# ================================================================
# 25. MAIN
# ================================================================

def main():

    verify_project()

    # ------------------------------------------------------------
    # Clean the three datasets explicitly mentioned in Day 2.
    # ------------------------------------------------------------

    nav = clean_nav_history()

    transactions = (
        clean_investor_transactions()
    )

    performance = (
        clean_scheme_performance()
    )

    # ------------------------------------------------------------
    # Copy the remaining seven datasets unchanged because the
    # Day-2 description does not specify transformations for them.
    # ------------------------------------------------------------

    copy_unchanged_datasets()

    # ------------------------------------------------------------
    # Create SQLite database/schema.
    # ------------------------------------------------------------

    engine = create_database()

    # ------------------------------------------------------------
    # Load dimensions.
    # ------------------------------------------------------------

    print()
    print("=" * 70)
    print("LOADING DIMENSIONS")
    print("=" * 70)

    # AUM is needed to build dim_date.
    aum = pd.read_csv(
        PROCESSED /
        "03_aum_by_fund_house.csv"
    )

    aum_date_column = find_column(
        aum,
        ["date"],
        label="date"
    )

    aum["date"] = parse_dates(
        aum[aum_date_column]
    )

    load_dim_date(
        engine,
        nav,
        transactions,
        performance,
        aum
    )

    load_dim_fund(
        engine,
        nav,
        transactions,
        performance
    )

    load_dim_investor(
        engine,
        transactions
    )

    # ------------------------------------------------------------
    # Load facts.
    # ------------------------------------------------------------

    print()
    print("=" * 70)
    print("LOADING FACT TABLES")
    print("=" * 70)

    nav_count = load_nav_fact(
        engine,
        nav
    )

    transaction_count = (
        load_transaction_fact(
            engine,
            transactions
        )
    )

    performance_count = (
        load_performance_fact(
            engine,
            performance
        )
    )

    aum_count = load_aum_fact(
        engine
    )

    # ------------------------------------------------------------
    # Verification.
    # ------------------------------------------------------------

    expected_counts = {
        "fact_nav": nav_count,
        "fact_transactions": transaction_count,
        "fact_performance": performance_count,
        "fact_aum": aum_count
    }

    verify_database(
        engine,
        expected_counts
    )

    # ------------------------------------------------------------
    # SQL files.
    # ------------------------------------------------------------

    write_queries_file()

    write_data_dictionary()

    # ------------------------------------------------------------
    # Run queries.
    # ------------------------------------------------------------

    run_queries(
        engine
    )

    # ------------------------------------------------------------
    # Finish.
    # ------------------------------------------------------------

    print()
    print("=" * 70)
    print("DAY 2 PIPELINE COMPLETE")
    print("=" * 70)

    print()
    print("Processed data:")
    print(
        PROCESSED
    )

    print()
    print("SQLite database:")
    print(
        DB_PATH
    )

    print()
    print("Schema:")
    print(
        SCHEMA_PATH
    )

    print()
    print("Queries:")
    print(
        QUERIES_PATH
    )

    print()
    print("Data dictionary:")
    print(
        DICTIONARY_PATH
    )

    print()
    print(
        'Git commit command:'
    )

    print(
        'git add . && git commit -m "Day 2: Cleaned data + SQLite DB loaded"'
    )


# ================================================================
# 26. ERROR HANDLING
# ================================================================

if __name__ == "__main__":

    try:

        main()

    except KeyError as error:

        print()
        print("=" * 70)
        print("STOPPED — COLUMN DETECTION ERROR")
        print("=" * 70)
        print(error)
        print()

        sys.exit(1)

    except FileNotFoundError as error:

        print()
        print("=" * 70)
        print("STOPPED — FILE NOT FOUND")
        print("=" * 70)
        print(error)
        print()

        sys.exit(1)

    except Exception:

        print()
        print("=" * 70)
        print("PIPELINE FAILED")
        print("=" * 70)

        traceback.print_exc()

        print()

        sys.exit(1)



BLUESTOCK MF - DAY 2 PIPELINE
PROJECT CHECK
Project root : C:\Users\javee\Bluestock internship
Raw data     : C:\Users\javee\Bluestock internship\data\raw
Processed    : C:\Users\javee\Bluestock internship\data\processed

[OK] 01_fund_master.csv
[OK] 02_nav_history.csv
[OK] 03_aum_by_fund_house.csv
[OK] 04_monthly_sip_inflows.csv
[OK] 05_category_inflows.csv
[OK] 06_industry_folio_count.csv
[OK] 07_scheme_performance.csv
[OK] 08_investor_transactions.csv
[OK] 09_portfolio_holdings.csv
[OK] 10_benchmark_indices.csv

[OK] All 10 datasets found.

PROCESSING OTHER DATASETS
[COPIED] 01_fund_master.csv                  40 rows
[COPIED] 03_aum_by_fund_house.csv            90 rows
[COPIED] 04_monthly_sip_inflows.csv          48 rows
[COPIED] 05_category_inflows.csv             144 rows
[COPIED] 06_industry_folio_count.csv         21 rows
[COPIED] 09_portfolio_holdings.csv           322 rows
[COPIED] 10_benchmark_indices.csv            8,050 rows

CLEANING 02_nav_history.csv
Source rows: 46,00

KeyError: "Required column not found.\nTried: ['year', 'reporting_year']\nAvailable columns: ['date', 'fund_house', 'aum_lakh_crore', 'aum_crore', 'num_schemes']"

In [2]:
 %pip install sqlalchemy

     ---------------------------------------- 0.0/10.5 MB ? eta -:--:--
     --- ------------------------------------ 1.0/10.5 MB 5.4 MB/s eta 0:00:02
     ---- ----------------------------------- 1.3/10.5 MB 3.5 MB/s eta 0:00:03
     ----- ---------------------------------- 1.6/10.5 MB 2.4 MB/s eta 0:00:04
     ------- -------------------------------- 2.1/10.5 MB 2.4 MB/s eta 0:00:04
     --------- ------------------------------ 2.6/10.5 MB 2.5 MB/s eta 0:00:04
     ------------ --------------------------- 3.4/10.5 MB 2.7 MB/s eta 0:00:03
     --------------- ------------------------ 4.2/10.5 MB 2.9 MB/s eta 0:00:03
     ------------------- -------------------- 5.2/10.5 MB 3.2 MB/s eta 0:00:02
     ---------------------- ----------------- 6.0/10.5 MB 3.2 MB/s eta 0:00:02
     ------------------------- -------------- 6.8/10.5 MB 3.2 MB/s eta 0:00:02
     ---------------------------- ----------- 7.6/10.5 MB 3.4 MB/s eta 0:00:01
     -------------------------------- ------- 8.7/10.5 MB 3